# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`

This notebook provides a guided template for loading and exploring the *A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya* using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata # the Dataset object (not dict)

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

`mlcroissant` lets you access record sets and fields defined in the Croissant schema. Below, we print information about each record set and its fields using their `@id` values.

In [ ]:
# Explore available record sets and fields:
record_sets = list(dataset.record_sets)
print("Record Set @id values and their fields:")
for rs in record_sets:
    print(f"Record Set: {rs['@id']} | Name: {rs.get('name', '')}")
    if 'fields' in rs and rs['fields']:
        for field in rs['fields']:
            print(f"  Field @id: {field['@id']}, Name: {field.get('name', '')}, DataType: {field.get('dataType', '')}")
    elif 'columns' in rs and rs['columns']:
        for col in rs['columns']:
            print(f"  Column @id: {col['@id']}, Name: {col.get('name', '')}, DataType: {col.get('dataType', '')}")
    else:
        print("  No fields or columns defined for this record set.")
print("\n")

# To demonstrate, print the first few record examples from the first record set (by @id):
if record_sets:
    example_record_set_id = record_sets[0]['@id']
    print(f"Previewing 3 records from record set @id: {example_record_set_id}")
    for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
        print(json.dumps(record, indent=2))
        if i >= 2:
            break


## 3. Data Extraction

Load data from each record set into a DataFrame for analysis.

Below, we extract all records from each record set using their `@id`.

In [ ]:
# Extract data from each record set
dataframes = {}

record_set_ids = [rs['@id'] for rs in record_sets]
print("Loading records for record sets:")
for rs_id in record_set_ids:
    print(f"  {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame([])
    dataframes[rs_id] = df

# Show columns for the first record set and preview the data
main_record_set_id = record_set_ids[0] if record_set_ids else None

if main_record_set_id:
    print(f"Columns in DataFrame for record set @id: {main_record_set_id}")
    print(dataframes[main_record_set_id].columns.tolist())
    print("Sample records:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll select a numeric field for demonstration. Make sure to use the field's `@id` as the column reference.

In [ ]:
# Identify a numeric field (e.g., GAD-7 score) from schema
numeric_field_id = None
group_field_id = None

# Heuristic: search for GAD-7, PHQ-9, PSQ or similar fields
fields_to_search = ['gad7_total', 'phq9_total', 'psq_score', 'age_of_participant', 'age', 'GAD7Total', 'PHQ9Total', 'PSQScore']

df = dataframes[main_record_set_id]
for col in df.columns:
    for candidate in fields_to_search:
        if candidate.lower() in col.lower():
            numeric_field_id = col
            break
    if numeric_field_id:
        break

# Find a group field (e.g., gender or education)
group_fields_candidates = ['gender', 'Gender', 'level_of_education', 'LevelOfEducation', 'marital_status', 'MaritalStatus', 'village', 'Village']
for col in df.columns:
    for candidate in group_fields_candidates:
        if candidate.lower() == col.lower():
            group_field_id = col
            break
    if group_field_id:
        break

if numeric_field_id:
    print(f"Using numeric field: {numeric_field_id}")
    threshold = 10
    numeric_values = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df[numeric_values > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (numeric_values - numeric_values.mean()) / numeric_values.std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        print(f"Grouping filtered records by: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        display(grouped_df.head())
    else:
        print("No group field found for grouping (e.g., gender or education).")
else:
    print("No numeric field found for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. For example, plot the distribution of the selected numeric field and compare group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
if numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna().astype(float), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# Plot mean values by group (e.g., by gender)
if group_field_id and numeric_field_id:
    plt.figure(figsize=(8, 5))
    means = df.groupby(group_field_id)[numeric_field_id].mean()
    sns.barplot(x=means.index, y=means.values)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.show()

## 6. Conclusion

- This notebook demonstrated a workflow for loading, exploring, and processing the *A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya* using `mlcroissant`.
- We accessed schema metadata, reviewed record sets and fields via their `@id` values, extracted survey data, and performed EDA including filtering, normalization, and grouping operations.
- Visualizations illustrated distributions and group comparisons.

For further analysis, consider exploring more fields, cross-correlations, or extending to predictive modeling using the dataset's AI-ready structure.